In [0]:
orders = [
    (1001, 101, 5000, "Created", "2026-08-18 09:00:00"),
    (1002, 102, 3000, "Created", "2026-08-18 09:05:00"),
    (1001, 101, 5000, "Shipped", "2026-08-18 10:00:00"),
    (1003, 103, 7000, "Created", "2026-08-18 10:10:00"),
    (1001, 101, 5000, "Delivered", "2026-08-18 11:00:00"),
    (1002, 102, 3000, "Shipped", "2026-08-18 11:30:00")
]

columns = [
    "OrderId",
    "CustomerId",
    "Amount",
    "Status",
    "UpdatedAt"
]

df=spark.createDataFrame(orders,columns)
display(df)

OrderId,CustomerId,Amount,Status,UpdatedAt
1001,101,5000,Created,2026-08-18 09:00:00
1002,102,3000,Created,2026-08-18 09:05:00
1001,101,5000,Shipped,2026-08-18 10:00:00
1003,103,7000,Created,2026-08-18 10:10:00
1001,101,5000,Delivered,2026-08-18 11:00:00
1002,102,3000,Shipped,2026-08-18 11:30:00


In [0]:
df.write.mode("overwrite").saveAsTable("bronze.day9_orders")

In [0]:
%sql
select * from bronze.day9_orders order by OrderId, UpdatedAt

OrderId,CustomerId,Amount,Status,UpdatedAt
1001,101,5000,Created,2026-08-18 09:00:00
1001,101,5000,Shipped,2026-08-18 10:00:00
1001,101,5000,Delivered,2026-08-18 11:00:00
1002,102,3000,Created,2026-08-18 09:05:00
1002,102,3000,Shipped,2026-08-18 11:30:00
1003,103,7000,Created,2026-08-18 10:10:00


### Data Quality rules:
- OrderId cannot be NULL
- CustomerId cannot be NULL
- Amount must be greater than 0
- Status cannot be NULL
- UpdatedAt cannot be NULL

In [0]:
from pyspark.sql.functions import col
invalid_df=df.filter(
    col("OrderId").isNull()
   |col("CustomerId").isNull()
   |(col("Amount")<=0)
   |col("Status").isNull()
   |col("UpdatedAt").isNull()
                     )
display(invalid_df)

OrderId,CustomerId,Amount,Status,UpdatedAt


In [0]:

valid_df=df.filter(
    col("OrderId").isNotNull()
   |col("CustomerId").isNotNull()
   |(col("Amount")>0)
   |col("Status").isNotNull()
   |col("UpdatedAt").isNotNull()
                     )
display(valid_df)

OrderId,CustomerId,Amount,Status,UpdatedAt
1001,101,5000,Created,2026-08-18 09:00:00
1002,102,3000,Created,2026-08-18 09:05:00
1001,101,5000,Shipped,2026-08-18 10:00:00
1003,103,7000,Created,2026-08-18 10:10:00
1001,101,5000,Delivered,2026-08-18 11:00:00
1002,102,3000,Shipped,2026-08-18 11:30:00


In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

window_spec=(
    Window
    .partitionBy("OrderId")
    .orderBy(col("UpdatedAt").desc())
)

ranked_df=valid_df.withColumn(
    "row_num",
    row_number().over(window_spec)
    )
display(ranked_df)

OrderId,CustomerId,Amount,Status,UpdatedAt,row_num
1001,101,5000,Delivered,2026-08-18 11:00:00,1
1001,101,5000,Shipped,2026-08-18 10:00:00,2
1001,101,5000,Created,2026-08-18 09:00:00,3
1002,102,3000,Shipped,2026-08-18 11:30:00,1
1002,102,3000,Created,2026-08-18 09:05:00,2
1003,103,7000,Created,2026-08-18 10:10:00,1


In [0]:
latest_orders=(
    ranked_df
    .filter(col("row_num")==1)
    .drop("row_num")
)

display(latest_orders)

OrderId,CustomerId,Amount,Status,UpdatedAt
1001,101,5000,Delivered,2026-08-18 11:00:00
1002,102,3000,Shipped,2026-08-18 11:30:00
1003,103,7000,Created,2026-08-18 10:10:00


In [0]:
latest_orders.write.mode("overwrite").saveAsTable("silver.day9_orders")

In [0]:
%sql
select * from silver.day9_orders order by OrderId;

OrderId,CustomerId,Amount,Status,UpdatedAt
1001,101,5000,Delivered,2026-08-18 11:00:00
1002,102,3000,Shipped,2026-08-18 11:30:00
1003,103,7000,Created,2026-08-18 10:10:00


In [0]:
new_orders = [
    (1001, 101, 5000, "Delivered", "2026-08-19 09:00:00"),
    (1002, 102, 3000, "Delivered", "2026-08-19 09:05:00"),
    (1004, 104, 8000, "Created", "2026-08-19 09:10:00")
]

new_df=spark.createDataFrame(new_orders,columns)
display(new_df)

OrderId,CustomerId,Amount,Status,UpdatedAt
1001,101,5000,Delivered,2026-08-19 09:00:00
1002,102,3000,Delivered,2026-08-19 09:05:00
1004,104,8000,Created,2026-08-19 09:10:00


In [0]:
new_df.createOrReplaceTempView("new_orders")

In [0]:
%sql
MERGE INTO silver.day9_orders as target
using new_orders as source
on target.OrderId= source.OrderId
when MATCHED THEN UPDATE SET *
when NOT MATCHED THEN INSERT *

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
3,2,0,1


In [0]:
%sql
SELECT * FROM silver.day9_orders order by OrderId

OrderId,CustomerId,Amount,Status,UpdatedAt
1001,101,5000,Delivered,2026-08-19 09:00:00
1002,102,3000,Delivered,2026-08-19 09:05:00
1003,103,7000,Created,2026-08-18 10:10:00
1004,104,8000,Created,2026-08-19 09:10:00
